# Chronological 28-Day Forecasting Experiments

Research: **Evaluating the Effectiveness of AI-Based Demand Forecasting for Retail Businesses Under Volatile Demand Conditions.** The original exploration notebook is unchanged. This notebook presents the completed full-population experiment and the earlier 5,000-series pilot.


## Experimental design

The full M5 validation dataset contains 30,490 item-store series. The experiment forecasts 28 days in each of three chronological folds. Each fold uses a rolling 365-day training window and chronological early stopping. Series sampling seed is 42 (the full run uses all series). Volatility groups are recomputed each fold from terciles of the coefficient of variation of positive-demand weekly totals in the prior 52 complete weeks.

Models: 7-day seasonal naive, four-week weekday mean, recursive LightGBM, and recursive LightGBM with scheduled prices. WAPE is pooled absolute error divided by pooled actual demand. Bootstrap intervals resample series IDs and retain each series' observations together (1,000 replicates).


In [ ]:
from pathlib import Path
import pandas as pd

# Prefer a newly generated full run; use the committed snapshot otherwise.
local = Path("artifacts_full_m5")
report = Path("reports/m5_full_population")
root = local if (local / "pooled_metrics.csv").exists() else report
fold_name = "metrics.csv" if root == local else "fold_metrics.csv"
folds = pd.read_csv(root / fold_name)
pooled = pd.read_csv(root / "pooled_metrics.csv")
bootstrap = pd.read_csv(root / "bootstrap_comparison.csv")
manifest = pd.read_json(root / "run_manifest.json", typ="series")
print(f"Results loaded from: {root}")
manifest


## Full-population pooled results

Each model has 2,561,160 forecast observations across three folds. Scores pool errors and actual demand across all observations.


In [ ]:
all_scores = pooled.loc[pooled["volatility_group"].eq("all")].copy()
all_scores[["model", "pooled_WAPE", "pooled_MAE", "folds", "forecast_observations"]].sort_values("pooled_WAPE")


## Performance under demand volatility

Compare price-aware LightGBM against the four-week weekday-mean baseline. Intervals are series-cluster bootstrap intervals and are conditional on this M5 data and model setup.


In [ ]:
group_ci = bootstrap.loc[
    bootstrap["comparison"].eq("baseline")
    & bootstrap["reference_model"].eq("four_week_weekday_mean")
    & bootstrap["candidate_model"].eq("lightgbm_recursive_price"),
    ["volatility_group", "series", "WAPE_improvement", "improvement_95CI_low", "improvement_95CI_high", "bootstrap_replicates"],
].sort_values("volatility_group")
group_ci


## Interpretation

Price-aware LightGBM achieved the lowest overall pooled WAPE. Its improvement over the weekday-mean baseline was larger in moderate- and high-volatility series. In low volatility, price-aware and no-price LightGBM were effectively tied; scheduled price features should not be assumed to help every subgroup.

## Saved model for inference

Run `python src/train_final_model.py --data-dir data/raw --output-dir artifacts/final_model --max-series 0` to choose the tree count with the final chronological early-stopping window and refit on the most recent 365 days across all series. It writes a local LightGBM model and metadata, which are ignored by Git. `src/forecast_inference.py` loads that bundle and forecasts one series from the latest 28 sales, 28 recent prices, and the future calendar and scheduled prices. The FastAPI service is now available at `src/api.py`: start it with `python -m uvicorn src.api:app --host 127.0.0.1 --port 8000`, then open `/docs` to submit a forecast request. Keep this research prototype local; it has no authentication.

These results use the M5 dataset and are not automatically generalizable to other retailers. This phase does not tune model hyperparameters, estimate uncertainty in predictions, or include an independent promotion signal.
